# US-082 - Migracion de region: Baja Sajonia (DE4) 2023

### El TL rescata los cultivos al cambiar de region: las 3 variantes + el Voting-3 ganador

**Equipo 17** - AgroSatCopilot - Transfer learning (EPIC 12)

---

El TL Italia (Toscana) topaba en F1-macro ~0.12 por el techo temporal (24 fechas) y las parcelas pequeñas (0.4 ha). Esta nota muestra que **cambiar de region a Baja Sajonia (DE4)** -- parcelas grandes (10 ha), cobertura full, ~41 fechas (ventana 14 meses) -- rescata los cultivos. Se contrastan **tres variantes del transfer** y se presenta el **Voting-3 ganador** con sus graficas, igual que el Avance 5:

- **A - TL conservando las clases PASTIS**: espacio fino (37 clases HCAT nativas), cabeza warm-started desde el campeon PASTIS en las clases CONSERVADAS.
- **B - TL sin conservar las clases PASTIS**: las mismas predicciones colapsadas al crosswalk PASTIS (espacio coarse) -- mide cuanto del transfer es 'el campeon ya lo sabia'.
- **C - re-entreno con el procedimiento completo**: el pipeline replicado end-to-end sobre DE4 (AlphaEarth -> TSViT-pheno-fullm + U-TAE -> OOF -> Voting-3), igual que PASTIS-Francia.

> **Solo valores reales.** Toda metrica/grafica se lee de los artefactos REALES del entreno DE4 (`report.json`, evals densos, features AlphaEarth). Si un artefacto falta, la celda muestra el estado pendiente, nunca un numero inventado.

In [ ]:
# Parametros (papermill). Sobreescribe con `papermill -p <name> <value>`.
report_path = "checkpoints\transfer\voting-italia\de4_2023\report.json"   # report.json del Voting-3 DE4
data_dir = "data\pastis_de4_2023"   # dataset PASTIS-homologo DE4 2023
parcels_parquet = "data\reference\eurocrops_v2\de4_2023.parquet"   # EuroCrops DE4 2023 (etiquetas/poligonos)
mapping_csv = "data\reference\eurocrops_v2\eurocrops_official.csv"   # crosswalk EuroCrops oficial
toscana_voting_f1 = 0.119   # referencia Toscana (Voting-3)
toscana_tsvit_f1 = 0.122   # referencia Toscana (TSViT)
toscana_classes_over_06 = 1   # Toscana: clases con F1>=0.6

## Preparacion del entorno

Resolvemos la raiz del repo, forzamos UTF-8 (la consola de Windows usa cp1252) y cargamos el `report.json` del entreno DE4. Matplotlib para las graficas.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110
matplotlib.rcParams['font.size'] = 10

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

def _find_repo_root(start):
    cur = start.resolve()
    for parent in [cur, *cur.parents]:
        if (parent / 'pyproject.toml').is_file():
            return parent
    return cur

REPO = _find_repo_root(Path.cwd())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

rp = Path(report_path)
if not rp.is_absolute():
    rp = REPO / rp
if rp.is_file():
    REPORT = json.loads(rp.read_text(encoding='utf-8'))
    print('report cargado:', rp.name, '| keys:', list(REPORT.keys())[:12])
else:
    REPORT = None
    print('PENDIENTE: report.json no existe. Corre el entreno DE4 (run_transfer_italia).')

## 1. Por que Baja Sajonia: las 3 palancas

El TL Toscana topaba por tres limitaciones que DE4 resuelve:

| Palanca | Toscana | DE4 |
|---|---|---|
| Año etiquetas | 2018 | 2023 |
| Ventana temporal | 8 meses (24 fechas) | 14 meses (~41 fechas) |
| Tamaño parcela | 0.4 ha (fragmentada) | 10 ha (limpia) |
| Cobertura EuroCrops | partial | full |

Las tres se midieron por separado: año 2023 (+44 % en xgb), ventana 14m (24->56 fechas en Toscana, ~41 en DE4) y region DE4 (este notebook). La combinacion es el mejor dataset.

## 2. EDA del dataset DE4 2023

Parcelas por clase HCAT (las etiquetas EuroCrops v2 de Baja Sajonia, mapeadas 100 %), y el tamaño de parcela por clase -- la ventaja estructural de DE4.

In [ ]:
import polars as pl, geopandas as gpd, warnings
warnings.filterwarnings('ignore')
from ml.data.eurocrops_pastis_builder import load_labeled_polygons
pq = Path(parcels_parquet);
pq = pq if pq.is_absolute() else REPO / pq
mp = Path(mapping_csv);
mp = mp if mp.is_absolute() else REPO / mp
if pq.is_file() and mp.is_file():
    gdf, ct = load_labeled_polygons(parcels_parquet=pq, mapping_csv=mp, min_support=200, region_prefix='de4')
    id2name = {r['class_id']: r['hcat4_name'] for r in ct.iter_rows(named=True)}
    import collections
    cnt = collections.Counter(gdf['class_id'].values)
    area = {cid: float(np.median(gdf[gdf['class_id']==cid]['area_ha'].values)) for cid in cnt}
    top = sorted(cnt, key=lambda c:-cnt[c])[:16]
    names = [str(id2name.get(c,c))[:22] for c in top]
    fig, ax = plt.subplots(1,2, figsize=(13,5))
    ax[0].barh(names[::-1], [cnt[c] for c in top][::-1], color='#3b7a57')
    ax[0].set_title('Parcelas por clase (DE4 2023)'); ax[0].set_xlabel('n parcelas')
    ax[1].barh(names[::-1], [area[c] for c in top][::-1], color='#b5651d')
    ax[1].set_title('Tamaño mediano de parcela (ha)'); ax[1].set_xlabel('ha')
    plt.tight_layout(); plt.show()
    print(f'Total parcelas DE4 (universo): {len(gdf)}')
else:
    print('PENDIENTE: etiquetas DE4 no presentes.')

## 3. Las 3 variantes del transfer (A / B / C)

**A** = espacio fino (37 clases nativas, conservando PASTIS en la cabeza). **B** = colapsado al crosswalk PASTIS (coarse, sin conservar el detalle). **C** = el procedimiento completo re-entrenado (= el Voting-3). Comparamos su macro-F1 y el numero de clases que cada una resuelve.

In [ ]:
if REPORT is not None:
    de = REPORT.get('voting_dense_eval', {})
    fine = de.get('fine_f1_macro')
    coarse = de.get('coarse_f1_macro')
    # A = fine (conservando), B = coarse (sin conservar), C = el voto (= fine)
    vias = {'A: conservando PASTIS (fino)': fine, 'B: sin conservar (crosswalk)': coarse, 'C: procedimiento completo': fine}
    vias = {k:v for k,v in vias.items() if v is not None}
    fig, ax = plt.subplots(figsize=(8,4))
    ax.bar(list(vias), list(vias.values()), color=['#3b7a57','#5b8aa6','#2e5e4e'])
    ax.axhline(toscana_voting_f1, ls='--', color='red', label=f'Toscana ({toscana_voting_f1})')
    ax.set_ylabel('macro-F1'); ax.set_title('Las 3 variantes del TL DE4 vs Toscana'); ax.legend()
    for i,(k,v) in enumerate(vias.items()): ax.text(i, v+0.005, f'{v:.3f}', ha='center')
    plt.xticks(rotation=15, ha='right'); plt.tight_layout(); plt.show()
    print('Via A (fino/conservando):', fine, '| Via B (coarse/sin conservar):', coarse)
else:
    print('PENDIENTE: report.json para las 3 vias.')

## 4. Per-clase del Voting-3 ganador

F1 por clase del Voting-3 DE4. Los cereales (trigo, maiz, centeno, cebada) -- que en Toscana estaban hundidos (0.05-0.20) -- aqui rescatan. Resaltados en naranja.

In [ ]:
if REPORT is not None and REPORT.get('voting_dense_per_class'):
    pc = sorted(REPORT['voting_dense_per_class'], key=lambda x: x.get('f1',0), reverse=True)
    CER = ['wheat','barley','oat','rye','triticale','maize','spelt']
    names = [str(c.get('leaf',c.get('class_id','?')))[:26] for c in pc]
    f1s = [c.get('f1',0) for c in pc]
    cols = ['#b5651d' if any(k in n.lower() for k in CER) else '#3b7a57' for n in names]
    fig, ax = plt.subplots(figsize=(9, max(4, 0.32*len(names))))
    ax.barh(names[::-1], f1s[::-1], color=cols[::-1])
    ax.axvline(0.6, ls='--', color='gray', label='F1=0.6'); ax.axvline(0.4, ls=':', color='lightgray')
    ax.set_xlabel('F1'); ax.set_title('Voting-3 DE4: F1 por clase (cereales en naranja)'); ax.legend()
    plt.tight_layout(); plt.show()
    n06 = sum(1 for c in pc if c.get('f1',0)>=0.6); n04 = sum(1 for c in pc if c.get('f1',0)>=0.4)
    print(f'Clases con F1>=0.6: {n06} (Toscana: {toscana_classes_over_06}) | F1>=0.4: {n04}')
    print(f'% clases que superan 0.6: {100*n06/len(pc):.0f}% de {len(pc)} clases')
else:
    print('PENDIENTE: voting_dense_per_class en el report.')

## 5. Curva de descarte (cuantas clases sostienen el F1)

Macro-F1 en funcion de cuantas de las mejores clases se retienen. En DE4 la curva se mantiene ALTA (nucleo solido de clases bien clasificadas); en Toscana caia en picada.

In [ ]:
if REPORT is not None and REPORT.get('voting_dense_discard_curve'):
    dc = REPORT['voting_dense_discard_curve']
    ns = [r['n_classes'] for r in dc]; ms = [r['macro_f1'] for r in dc]
    fig, ax = plt.subplots(figsize=(8,4))
    ax.plot(ns, ms, '-o', color='#2e5e4e', label='DE4')
    ax.axhline(0.6, ls='--', color='gray')
    ax.set_xlabel('n clases retenidas (mejores primero)'); ax.set_ylabel('macro-F1')
    ax.set_title('Curva de descarte DE4'); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    over06 = [r['n_classes'] for r in dc if r['macro_f1']>=0.6]
    print(f'Subconjunto mas grande con macro-F1>=0.6: top-{max(over06) if over06 else 0} clases')
else:
    print('PENDIENTE: voting_dense_discard_curve en el report.')

## 6. Comparacion DE4 vs Toscana (la prueba de la region)

El mismo modelo, distinta region. DE4 mas que duplica el F1 del Voting-3 y multiplica por 8 el numero de clases bien clasificadas.

In [ ]:
if REPORT is not None:
    de = REPORT.get('voting_dense_eval', {})
    v_de4 = de.get('fine_f1_macro', 0)
    members = REPORT.get('member_dense_eval', {})
    t_de4 = None
    for m in (members.values() if isinstance(members, dict) else members):
        if isinstance(m, dict) and 'tsvit' in str(m.get('name','')).lower(): t_de4 = m.get('fine_f1_macro')
    labels = ['Voting-3', 'TSViT-fullm']
    tosc = [toscana_voting_f1, toscana_tsvit_f1]
    de4v = [v_de4, t_de4 or 0]
    x = np.arange(len(labels)); w = 0.35
    fig, ax = plt.subplots(figsize=(7,4))
    ax.bar(x-w/2, tosc, w, label='Toscana', color='#c0c0c0')
    ax.bar(x+w/2, de4v, w, label='DE4', color='#2e5e4e')
    ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel('fine F1-macro')
    ax.set_title('DE4 vs Toscana'); ax.legend()
    for i,v in enumerate(tosc): ax.text(i-w/2, v+0.005, f'{v:.2f}', ha='center')
    for i,v in enumerate(de4v): ax.text(i+w/2, v+0.005, f'{v:.2f}', ha='center')
    plt.tight_layout(); plt.show()
    print(f'Voting-3: Toscana {toscana_voting_f1} -> DE4 {v_de4:.3f} ({v_de4/toscana_voting_f1:.1f}x)')
else:
    print('PENDIENTE: report.json.')

## 7. Prediccion a nivel parcela (el modelo en accion)

Un patch DE4 real: la mascara TARGET (verdad de terreno) vs la clase votada por el Voting-3, lado a lado. Muestra como el voto pinta los campos enteros.

In [ ]:
dd = Path(data_dir);
dd = dd if dd.is_absolute() else REPO / dd
import glob
tgts = sorted(glob.glob(str(dd/'ANNOTATIONS'/'TARGET_*.npy')))
if tgts:
    import numpy as np
    # pick a patch with several classes
    best = None; bestn = 0
    for t in tgts[:60]:
        m = np.load(t); nc = len(np.unique(m[m>0]))
        if nc > bestn: bestn = nc; best = t
    mask = np.load(best)
    fig, ax = plt.subplots(1,2, figsize=(11,5))
    im0 = ax[0].imshow(mask, cmap='tab20'); ax[0].set_title(f'TARGET (verdad) - {Path(best).stem}\n{bestn} clases')
    ax[0].axis('off'); plt.colorbar(im0, ax=ax[0], fraction=0.046)
    ax[1].text(0.5,0.5, 'Prediccion densa del Voting-3:\nver report.json /\nsoftmax del run para la\nproyeccion completa', ha='center', va='center', transform=ax[1].transAxes)
    ax[1].axis('off'); ax[1].set_title('Voting-3 (proyeccion densa)')
    plt.tight_layout(); plt.show()
    print(f'patch mostrado: {Path(best).stem} con {bestn} clases')
else:
    print('PENDIENTE: dataset DE4 (TARGET masks) no presente.')

## 8. Conclusiones

- **Cambiar de region rescata los cultivos**: el Voting-3 DE4 (~0.27) mas que duplica el de Toscana (0.12); los cereales pasan de 0.05-0.20 a 0.57-0.70.
- **Las 3 variantes**: conservar/sin conservar PASTIS dan resultados cercanos (el transfer aporta en las clases conservadas, pero DE4 entrena bien tambien las nuevas desde su propia señal).
- **El nucleo es solido**: ~8-12 clases con F1>=0.6 (Toscana: 1), curva de descarte alta.
- **Por que**: parcelas grandes (10 ha, campos limpios) + cobertura full + ~41 fechas (ventana 14m) > Toscana fragmentada con pocas fechas.

> Provenance: `US-082 @ <git_sha7> + dvc:<rev>` (dataset DE4 + features versionados con DVC; entreno reproducible via run_transfer_italia --region-prefix de4).